# Protein language models: ESM-2, embeddings and mutation scores

A **protein language model (pLM)** learns patterns in amino-acid sequences.

**ESM-2** is a transformer trained to predict masked amino acids from the surrounding sequence. We will inspect its model and tokenizer, then calculate last-layer embeddings, visualize them with UMAP, and score amino-acid substitutions.

Complete the numbered exercises. The first task is to find the specifications of the selected checkpoint.

[Open this branch in Colab](https://colab.research.google.com/github/nbrg-ppcu/appliedmedtech/blob/session-08-protein-language-models/notebooks/session_08/ProteinLM_practice.ipynb) · [Published version after merge](https://colab.research.google.com/github/nbrg-ppcu/appliedmedtech/blob/main/notebooks/session_08/ProteinLM_practice.ipynb)


## Amino acids and their codes

Protein sequences usually use **20 standard amino acids**, represented by one-letter codes. An amino acid within a protein chain is called a **residue**.

| One-letter code | Three-letter code | Amino acid |
|---|---|---|
| A | Ala | Alanine |
| C | Cys | Cysteine |
| D | Asp | Aspartic acid |
| E | Glu | Glutamic acid |
| F | Phe | Phenylalanine |
| G | Gly | Glycine |
| H | His | Histidine |
| I | Ile | Isoleucine |
| K | Lys | Lysine |
| L | Leu | Leucine |
| M | Met | Methionine |
| N | Asn | Asparagine |
| P | Pro | Proline |
| Q | Gln | Glutamine |
| R | Arg | Arginine |
| S | Ser | Serine |
| T | Thr | Threonine |
| V | Val | Valine |
| W | Trp | Tryptophan |
| Y | Tyr | Tyrosine |

Some proteins also contain **selenocysteine (`U`)** or **pyrrolysine (`O`)**. Ambiguity codes are different: `B` means D or N, `Z` means E or Q, and `X` means an unspecified amino acid.


## Setup

1. Open in Google Colab and choose **File → Save a copy in Drive**.
2. Select **Runtime → Change runtime type → Python 3 → GPU** (T4 is sufficient).
3. Run the installation cell before the imports, then work through the cells in order.

The model and dataset download directly from Hugging Face; no manual uploads, Drive mount, or Hugging Face token are required. An internet connection is needed for the first download. Colab includes PyTorch and the scientific Python packages used here.

CPU execution is supported but slower. For a quick CPU check, change `N_PROTEINS = 1000` to `N_PROTEINS = 100` in the selection cell. Use at least 51 proteins for Exercise 9.

If packages were imported before installation, choose **Runtime → Restart session**, then rerun from the top. A missing `HF_TOKEN` warning for these public resources can be ignored.

Exercise cells are intentionally incomplete; the supplied examples can run in order without filling in the answers.


In [ ]:
%pip -q install "transformers==4.57.1" "datasets>=3,<5" "umap-learn==0.5.9.post2" seaborn


In [ ]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM
from datasets import load_dataset
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from tqdm.auto import tqdm


## Load the model and tokenizer

The **tokenizer** converts amino-acid letters into token IDs. The **model** processes these IDs.


In [ ]:
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
# A short fragment from the original example.
protein_sequence = "MAPLRKTYVLKL"
model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)


In [ ]:
model


`esm` is the transformer backbone. `lm_head` converts its representations into scores for tokens.


### Exercise 1 — Find the model specifications

Use the links below and inspect `model.config` to find:

- the number of transformer layers;
- the embedding size (`hidden_size`);
- the approximate number of parameters;
- the vocabulary size.

What do `t6` and `8M` mean in the checkpoint name? Count the parameters in the loaded model and compare the count with the model card.

Sources: [model card](https://huggingface.co/facebook/esm2_t6_8M_UR50D) · [checkpoint configuration](https://huggingface.co/facebook/esm2_t6_8M_UR50D/blob/main/config.json) · [ESM documentation](https://huggingface.co/docs/transformers/v4.57.2/en/model_doc/esm).

Hint: each parameter tensor has a `.numel()` method; `model.parameters()` iterates over the parameter tensors.


In [ ]:
# Inspect model.config and record the requested values.
# Calculate the total number of parameters.


In [ ]:
tokenizer


## Tokenize one protein fragment

The 12-residue fragment keeps the tokens readable. Each amino acid becomes one token, with a start token and an end token added by the tokenizer.


In [ ]:
input_ids = tokenizer.encode(protein_sequence, return_tensors="pt")
input_ids


In [ ]:
print("Amino acids:", len(protein_sequence))
print("Input shape:", input_ids.shape)


The shape is **[batch size, number of tokens]**. Here, the fragment contains 12 amino acids and becomes 14 tokens.


In [ ]:
tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
tokens


## Special tokens and IDs

`<cls>` marks the start, `<eos>` marks the end, `<pad>` fills shorter sequences, and `<mask>` hides a residue. `<unk>` represents an unknown token.

Token IDs are lookup indices, not measures of amino-acid similarity. `X` is a residue symbol; `<unk>` is the tokenizer's fallback for an unsupported symbol.


In [ ]:
special_tokens = tokenizer.all_special_tokens
for special_token in special_tokens:
    print(f"{special_token:10} ID: {tokenizer.convert_tokens_to_ids(special_token)}")


In [ ]:
for amino_acid in amino_acids:
    print(f"{amino_acid}  ID: {tokenizer.convert_tokens_to_ids(amino_acid)}")


In [ ]:
aa2esmid = {aa: tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids}
aa2esmid


### Exercise 2 — Read and decode tokens

Tokenize the first six residues. Print the IDs, convert them back to tokens, and count the tokens. Find the ID of `M` and explain the two extra tokens.


In [ ]:
ex2_fragment = protein_sequence[:6]
# Tokenize ex2_fragment, inspect its IDs and decode them.


### Exercise 3 — Inspect padding

Tokenize the 12-residue fragment together with its first six residues. Use `padding=True` and `return_tensors="pt"`.

Print the input IDs and attention mask. How many padding tokens are added to the shorter sequence? The attention mask is 1 for input tokens and 0 for padding.


In [ ]:
ex3_sequences = [protein_sequence, protein_sequence[:6]]
# Tokenize both sequences together and inspect the result.


## One forward pass

A **forward pass** calculates the model's outputs from its current parameters:

1. Token IDs select numerical vectors from an embedding table.
2. The transformer layers update each vector using the sequence context.
3. The language-model head converts each final vector into scores for possible tokens.

Training adds a loss calculation, backpropagation and an optimizer step. Here we inspect the outputs of the forward pass.

`eval()` turns off training behavior such as dropout. `no_grad()` avoids storing calculations for gradients. `model(...)` returns token scores. For embeddings, we will use the backbone's last-layer output.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

with torch.no_grad():
    outputs = model(input_ids.to(device))

outputs.keys()


In [ ]:
outputs.logits.shape


**Logits:** [batch size, tokens, vocabulary size] = **[1, 14, 33]**.

`outputs.logits[0, 5, :]` contains 33 scores for possible tokens **at token position 5** of the first sequence. For this unpadded input, position 0 is `<cls>`, so token position 5 corresponds to residue 5, `R`.

A **logit** is an unnormalized score from the model's prediction head. It can be any real number. A higher score means a higher probability after softmax; the score itself is not a probability or a log-probability.

For one position, softmax converts its 33 scores into probabilities:

$$p_j = \frac{e^{z_j}}{\sum_{k=1}^{33} e^{z_k}}.$$

Normalize over the vocabulary dimension with `dim=-1`. ESM-2 predicts a token at the supplied position, rather than the next token in a sequence.

**Why 33 rather than 20?** The vocabulary contains 20 standard amino-acid codes, `X/B/U/Z/O`, `.` and `-`, and six special or reserved symbols.


In [ ]:
position = 5
position_logits = outputs.logits[0, position, :]
position_probabilities = torch.softmax(position_logits, dim=-1)
logit_table = pd.DataFrame({
    "token": tokenizer.convert_ids_to_tokens(list(range(model.config.vocab_size))),
    "logit": position_logits.detach().cpu().numpy(),
    "probability": position_probabilities.detach().cpu().numpy(),
})
print("Probability sum:", position_probabilities.sum().item())
logit_table.sort_values("probability", ascending=False).head(8)


The input above still contains `R` at residue 5: the model can see it. This shows the output format, but it is not a test of predicting a missing residue. Hide that residue in the next exercise.


### Exercise 4 — Predict a masked amino acid

Copy the tokenized fragment and replace residue 5 with `tokenizer.mask_token_id`. Run a forward pass, apply softmax at that position, and print the three most probable standard amino acids.

Also report the probability of the original residue, `R`. Do not change the original sequence or its token IDs.


In [ ]:
ex4_position = 5
ex4_ids = tokenizer.encode(protein_sequence, return_tensors="pt").clone()
# Replace the token at ex4_position with tokenizer.mask_token_id.
# Store this forward pass in ex4_output, then inspect its probabilities.


In [ ]:
with torch.no_grad():
    last_hidden_state = model.base_model(input_ids.to(device)).last_hidden_state

last_hidden_state.shape


`model.base_model(...)` runs the transformer without the token-prediction head. Its `last_hidden_state` contains the **last layer's embedding for every token**.

The shape is **[batch size, tokens, hidden size]**. These are contextual embeddings: a residue's vector depends on the surrounding sequence. Compare the last dimension with `model.config.hidden_size`.


In [ ]:
# Remove the batch dimension and the start/end tokens.
residue_embeddings = last_hidden_state[0, 1:-1, :]
residue_embeddings.shape


In [ ]:
# First ten values of the first amino acid's embedding.
residue_embeddings[0, :10]


**Mean pooling** averages the amino-acid vectors into one vector for the whole protein.


In [ ]:
protein_embedding = residue_embeddings.mean(dim=0)
protein_embedding.shape


### Exercise 5 — Read the tensor dimensions

Print the shape of the logits, last hidden state, residue embeddings and pooled protein vector. Extract the first eight values of residue 5's final embedding.

Explain the difference between the vocabulary size and the hidden size. Which dimension does mean pooling remove?


In [ ]:
# Inspect the shapes and extract residue 5's embedding values.


## Protein localization dataset

Each row is one protein. The **input** is its amino-acid sequence; the **label** is its location in the cell.


In [ ]:
dataset = load_dataset("neuralbioinfo/Course-ProteinLoc")
dataset


`dataset` is a `DatasetDict`: a collection of named splits. `dataset["train"]` selects one split, and `dataset["train"][0]` selects its first row.


In [ ]:
dataset["train"].features


In [ ]:
dataset["train"][0]


| Column | Meaning |
|---|---|
| `Protein ID` | Protein identifier |
| `Sequence` | Amino-acid sequence: the model input |
| `Localization` | Location in the cell: the label |
| `Full Header` | Original sequence header; contains label information |
| `Is Test` | Whether the protein is marked for testing |

The split is called `train`, but some rows have `Is Test=True`. We reserve those rows for later evaluation. Only `Sequence` is passed to the model.


In [ ]:
example = dataset["train"][0]
print("Protein:", example["Protein ID"])
print("Sequence length:", len(example["Sequence"]))
print("First 60 amino acids:", example["Sequence"][:60])
print("Label:", example["Localization"])


In [ ]:
label_counts = pd.Series(dataset["train"]["Localization"]).value_counts()
label_counts


### Exercise 6 — Describe the dataset

Inspect three rows. Print each protein's ID, sequence length and localization. Count the localization classes and find the most frequent class.

State what one sample is, which column is the input, and which is the label. Why should `Full Header` not be passed to the model?


In [ ]:
# Inspect three rows and calculate the class counts.


## Select and tokenize the proteins

For this plot, use up to 1,000 proteins that are not marked for testing. `filter` selects rows, `shuffle` changes their order, and `select` takes rows by index.


In [ ]:
N_PROTEINS = 1000
embedding_dataset = dataset["train"].filter(lambda row: not row["Is Test"])
embedding_dataset = embedding_dataset.shuffle(seed=42)
embedding_dataset = embedding_dataset.select(range(min(N_PROTEINS, len(embedding_dataset))))
embedding_dataset


In [ ]:
# Counts for the proteins that will actually appear in the plot.
pd.Series(embedding_dataset["Localization"]).value_counts()


Classes may have different numbers of points. A rare class can look less prominent simply because fewer examples were sampled.


`map` applies a function to the dataset. With `batched=True`, the function receives several sequences at once.

We pad or truncate each input to **512 tokens**, including the two boundary tokens. Proteins longer than 510 amino acids are shortened.

For a long protein, its embedding will describe only the retained first 510 residues. Information at the other end is lost.


In [ ]:
MAX_LENGTH = 512

def tokenize_function(examples):
    return tokenizer(
        examples["Sequence"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_special_tokens_mask=True,
    )

tokenized_dataset = embedding_dataset.map(tokenize_function, batched=True)
tokenized_dataset


Tokenization adds three columns:

- `input_ids`: token IDs.
- `attention_mask`: 1 for real input tokens, 0 for padding.
- `special_tokens_mask`: marks special tokens, which we will exclude from pooling.

Boundary tokens have attention-mask value 1, so the attention mask alone is not enough for amino-acid pooling.


In [ ]:
example = tokenized_dataset[0]
pd.DataFrame({
    "token": tokenizer.convert_ids_to_tokens(example["input_ids"]),
    "input_id": example["input_ids"],
    "attention_mask": example["attention_mask"],
    "special_token": example["special_tokens_mask"],
})


In [ ]:
lengths = np.array([len(sequence) for sequence in embedding_dataset["Sequence"]])
print("Proteins:", len(lengths))
print("Proteins truncated:", np.sum(lengths > MAX_LENGTH - 2))


## One batch

A `DataLoader` groups rows into batches. Convert the numeric columns to PyTorch tensors and keep the localization labels as text.


In [ ]:
tokenized_dataset = tokenized_dataset.select_columns([
    "input_ids", "attention_mask", "special_tokens_mask", "Localization"
])
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "special_tokens_mask"],
    output_all_columns=True,
)

batch_size = 16
dataloader = DataLoader(tokenized_dataset, batch_size=batch_size, shuffle=False)
batch = next(iter(dataloader))
batch.keys()


In [ ]:
print("Token IDs:", batch["input_ids"].shape)
print("Attention mask:", batch["attention_mask"].shape)
print("Labels:", batch["Localization"])


For embeddings, use `model.base_model`. This runs the transformer without the amino-acid prediction head.


In [ ]:
batch_input_ids = batch["input_ids"].to(device)
batch_attention_mask = batch["attention_mask"].to(device)

with torch.no_grad():
    batch_outputs = model.base_model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)

batch_outputs.keys()


In [ ]:
hidden_states = batch_outputs.last_hidden_state
hidden_states.shape


The shape is **[batch size, token positions, hidden size]**. Check that the first two dimensions match the input batch and that the last matches the model configuration.

Average only the amino-acid positions. Exclude padding, start tokens, and end tokens.


In [ ]:
residue_mask = batch["attention_mask"].bool() & ~batch["special_tokens_mask"].bool()
residue_mask = residue_mask.to(device)
residue_mask.sum(dim=1)


`unsqueeze(-1)` adds a dimension so the mask can be applied to every embedding value. `dim=1` is the token dimension.


In [ ]:
weights = residue_mask.unsqueeze(-1)
mean_pooled = (hidden_states * weights).sum(dim=1) / weights.sum(dim=1)
mean_pooled.shape


The result has shape **[batch size, hidden size]**: one vector per protein.


### Exercise 7 — Check the pooling mask

For one batch, calculate both the residue-only mean and the mean over all 512 token positions. Print the number of residues per protein and the largest absolute difference between the two vectors for each protein.

Explain why padding and boundary tokens should not contribute to a protein's mean embedding.


In [ ]:
# Compare mean_pooled with hidden_states.mean(dim=1).
# Report one difference value per protein.


## Multiple batches

Repeat the same calculation for every batch. Keep each protein's label next to its embedding.


In [ ]:
embeddings = []
labels = []

with torch.no_grad():
    for batch in tqdm(dataloader):
        batch_input_ids = batch["input_ids"].to(device)
        batch_attention_mask = batch["attention_mask"].to(device)
        batch_outputs = model.base_model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)
        hidden_states = batch_outputs.last_hidden_state

        residue_mask = batch["attention_mask"].bool() & ~batch["special_tokens_mask"].bool()
        weights = residue_mask.to(device).unsqueeze(-1)
        mean_pooled = (hidden_states * weights).sum(dim=1) / weights.sum(dim=1)

        embeddings.extend(mean_pooled.cpu().numpy())
        labels.extend(batch["Localization"])

embeddings = np.array(embeddings)
labels = np.array(labels)


In [ ]:
print("Embeddings:", embeddings.shape)
print("Labels:", labels.shape)
assert len(embeddings) == len(labels) == len(embedding_dataset)


### Exercise 8 — Handle the last batch

Reuse the embedding loop with **10 proteins** and **batch size 4**. Predict the batch sizes before running it. Store these results in separate variables.

Check that the final array has one vector per protein and that the labels remain in the same order as the selected rows.


In [ ]:
ex8_dataset = tokenized_dataset.select(range(10))
ex8_loader = DataLoader(ex8_dataset, batch_size=4, shuffle=False)
# Reuse the embedding loop with ex8_loader and new result lists.


## UMAP: a two-dimensional view

UMAP maps the protein vectors to two dimensions. Each point is one protein.

`n_neighbors` controls the neighborhood size. `min_dist` controls how tightly points can be packed. Labels provide the colors; they are not used to calculate the coordinates.


In [ ]:
umap_model = umap.UMAP(
    n_neighbors=15,
    min_dist=0.3,
    n_components=2,
    random_state=42,
)
umap_embeddings = umap_model.fit_transform(embeddings)
umap_embeddings.shape


In [ ]:
plt.figure(figsize=(10, 7))
unique_labels = np.unique(labels)
palette = sns.color_palette("tab10", n_colors=len(unique_labels))

for label, color in zip(unique_labels, palette):
    indices = labels == label
    plt.scatter(umap_embeddings[indices, 0], umap_embeddings[indices, 1],
                label=label, color=color, s=15, alpha=0.7)

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("Protein embeddings by localization")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()


### Exercise 9 — Compare two UMAP views

Create two plots using `n_neighbors=5` and `n_neighbors=50`. Keep the embeddings, `min_dist=0.3` and `random_state=42` fixed. Use the same label colors in both plots.

Name one observation that remains stable and one that changes. Does a more separated plot establish better localization prediction?


In [ ]:
# Fit two UMAP projections and compare the colored plots.
# Add two short observations in a Markdown cell.


## Pseudo-log-likelihood and amino-acid replacements

The embeddings describe a protein. We now return to the **language-model head** and ask how well it predicts each amino acid from the others.

For a sequence of length $L$, create $L$ copies. In copy $i$, hide only residue $i$, then read the probability assigned to its original amino acid. The other residues remain visible.

We will use the **full 108-residue sequence** from the original example. The model and tokenizer are already loaded.


In [ ]:
mutation_sequence = "MAPLRKTYVLKLYVAGNTPNSVRALKTLNNILEKEFKGVYALKVIDVLKNPQLAEEDKILATPTLAKVLPPPVRRIIGDLSNREKVLIGLDLLYEEIGDQAEDDLGLE"
scoring_input_ids = tokenizer.encode(mutation_sequence, return_tensors="pt")
sequence_length = len(mutation_sequence)
start_pos, end_pos = 1, sequence_length

# One start token, one token per residue, and one end token.
assert scoring_input_ids.shape[1] == sequence_length + 2
assert scoring_input_ids[0, 1:-1].tolist() == [aa2esmid[aa] for aa in mutation_sequence]
print("Residues:", sequence_length)
print("Tokenized shape:", scoring_input_ids.shape)


### Mask one position at a time

For this unpadded ESM input, token 0 is `<cls>`. Therefore **residue 1 is token 1**, while Python string index 0 stores its amino-acid letter.

`clone()` makes an independent copy. We do not change `scoring_input_ids` or the original sequence.


In [ ]:
input_ids_list = []
masked_positions = []

for position in range(start_pos, end_pos + 1):
    masked_input_ids = scoring_input_ids.clone()
    masked_input_ids[0, position] = tokenizer.mask_token_id
    input_ids_list.append(masked_input_ids[0])
    masked_positions.append(position)

batched_input_ids = torch.stack(input_ids_list)
masked_positions = torch.tensor(masked_positions)
print("Masked inputs:", batched_input_ids.shape)
print("First two masked positions:", masked_positions[:2])


### Exercise 10 — Check the masked copies

Decode the first two copies and locate their masks. Count the masks in every row and check that the original `scoring_input_ids` still contains no mask.

Explain why the row for residue 5 is `batched_input_ids[4]`, but its masked token is at index 5. Does this procedure hide several residues in the same copy?


In [ ]:
# Decode the first two rows and count masks per row.
# Check the original input and inspect the copy for residue 5.


### Dataset and DataLoader

Keep each masked sequence together with its masked position. The `DataLoader` returns both in a batch. We use `shuffle=False` so the results remain in protein-position order.


In [ ]:
from torch.utils.data import Dataset

class ProteinDataset(Dataset):
    def __init__(self, data, positions):
        self.data = data
        self.positions = positions

    def __len__(self):
        return self.data.size(0)

    def __getitem__(self, idx):
        return self.data[idx], self.positions[idx]

protein_ds = ProteinDataset(batched_input_ids, masked_positions)
mutation_batch_size = 8
mutation_loader = DataLoader(protein_ds, batch_size=mutation_batch_size, shuffle=False)
print("Masked sequences:", len(protein_ds))


### Read only the masked-site logits

The model returns logits for every token in each copy, but we need only its **masked position**. `rows` selects the correct position separately for every item in the batch.

After concatenation, row `position - 1` contains the vocabulary scores for that protein position. There are **108 rows and 33 vocabulary columns**, rather than a stored prediction tensor for every position in every copy.


In [ ]:
logits_ls = []
model.to(device)
model.eval()

with torch.no_grad():
    for batch_ids, batch_positions in tqdm(mutation_loader):
        batch_ids = batch_ids.to(device)
        batch_positions = batch_positions.to(device)
        output = model(batch_ids)
        rows = torch.arange(batch_ids.shape[0], device=device)
        masked_logits = output.logits[rows, batch_positions, :]
        logits_ls.append(masked_logits.cpu())
        del output, masked_logits

all_logits = torch.cat(logits_ls, dim=0)
print("Masked-site logits:", all_logits.shape)
assert all_logits.shape == (sequence_length, model.config.vocab_size)


### From logits to pseudo-log-likelihood

`log_softmax` calculates log probabilities stably. Normalize across the **complete vocabulary**, then select the original amino acid at each site:

$$\mathrm{PLL}(x)=\sum_{i=1}^{L}\log p_\theta(x_i\mid x_{\setminus i}).$$

The **mean PLL** divides this sum by $L$. Higher (less negative) values mean that the model assigns higher average log probability to the observed residues when they are hidden.

This is a **pseudo**-likelihood: the overlapping conditional predictions do not form the usual chain-rule factorization of a sequence probability. It is also not a measurement of biological function. The sum depends strongly on length; the mean makes the length normalization explicit.


In [ ]:
log_probabilities = torch.log_softmax(all_logits.float(), dim=-1)
assert torch.allclose(log_probabilities.exp().sum(dim=-1),
                      torch.ones(sequence_length), atol=1e-5)

wt_log_probabilities = []
for position in range(start_pos, end_pos + 1):
    wt_residue = mutation_sequence[position - 1]
    wt_token_id = aa2esmid[wt_residue]
    log_prob_wt = log_probabilities[position - 1, wt_token_id].item()
    wt_log_probabilities.append(log_prob_wt)

wt_log_probabilities = np.array(wt_log_probabilities)
pll = wt_log_probabilities.sum()
mean_pll = wt_log_probabilities.mean()
print(f"Summed PLL: {pll:.3f}")
print(f"Mean PLL per residue: {mean_pll:.3f}")

position_scores = pd.DataFrame({
    "position": range(1, sequence_length + 1),
    "original": list(mutation_sequence),
    "log_probability": wt_log_probabilities,
})
position_scores.head()


### Exercise 11 — Inspect the sequence score

Find the three positions with the lowest and the three with the highest original-residue log probability. Confirm that `pll / sequence_length` equals `mean_pll`.

Which residues were hardest for this model to recover? Explain why a higher summed PLL alone is not a fair comparison between proteins of very different lengths.


In [ ]:
# Sort position_scores by log_probability in both directions.
# Check the relation between summed and mean PLL.


### Score one amino-acid replacement

At the **same masked position**, compare a proposed replacement with the original residue:

$$\mathrm{LLR}_i(a)=\log p_\theta(a\mid x_{\setminus i})-\log p_\theta(x_i\mid x_{\setminus i}).$$

Positive means the model prefers the replacement; negative means it prefers the original. Replacing a residue with itself gives zero. Both terms use the **same original sequence context**, so no new forward pass is needed.

This single-site LLR is **not generally** `PLL(mutant) - PLL(original)`: changing one residue can alter predictions at all the other positions. A full mutant PLL would require masking and scoring every position again in the mutant sequence.


In [ ]:
position = 5
replacement = "K"
assert 1 <= position <= sequence_length and replacement in aa2esmid

wt_residue = mutation_sequence[position - 1]
log_prob_wt = log_probabilities[position - 1, aa2esmid[wt_residue]].item()
log_prob_mt = log_probabilities[position - 1, aa2esmid[replacement]].item()
llr = log_prob_mt - log_prob_wt
print(f"Substitution: {wt_residue}{position}{replacement}")
print(f"Original log probability: {log_prob_wt:.3f}")
print(f"Replacement log probability: {log_prob_mt:.3f}")
print(f"LLR: {llr:.3f}")


### All standard replacements at every position

Construct the same comparison for all 20 standard amino acids at every site. Rows are candidate replacements; columns are original protein positions.

The probabilities were normalized over all 33 vocabulary entries. We select the 20 standard residues **after** normalization for this heatmap.


In [ ]:
aa2heatmap_idx = {aa: idx for idx, aa in enumerate(amino_acids)}
heatmap = np.zeros((len(amino_acids), sequence_length))

for position in range(start_pos, end_pos + 1):
    wt_residue = mutation_sequence[position - 1]
    log_prob_wt = log_probabilities[position - 1, aa2esmid[wt_residue]].item()
    for aa in amino_acids:
        log_prob_mt = log_probabilities[position - 1, aa2esmid[aa]].item()
        heatmap[aa2heatmap_idx[aa], position - 1] = log_prob_mt - log_prob_wt

wt_rows = [aa2heatmap_idx[aa] for aa in mutation_sequence]
assert heatmap.shape == (20, sequence_length)
assert np.allclose(heatmap[wt_rows, np.arange(sequence_length)], 0.0)
print("Heatmap shape:", heatmap.shape)


In [ ]:
color_limit = max(1.0, float(np.abs(heatmap).max()))
plt.figure(figsize=(15, 5))
plt.imshow(heatmap, cmap="RdBu_r", aspect="auto", interpolation="nearest",
           vmin=-color_limit, vmax=color_limit)
tick_positions = list(range(1, sequence_length + 1, 10))
plt.xticks([p - 1 for p in tick_positions], tick_positions)
plt.yticks(range(len(amino_acids)), amino_acids)
plt.xlabel("Protein position (1-based)")
plt.ylabel("Replacement amino acid")
plt.title("Replacement LLRs: blue = lower than original; red = higher")
plt.colorbar(label="Log probability of replacement − log probability of original")
plt.tight_layout()
plt.show()


### Exercise 12 — Compare replacements and zoom in

Compare **R5K** with **R5W** using the same masked-position predictions. Which replacement receives the higher LLR?

Plot the first 10 heatmap columns. Label each column with its 1-based position and original amino acid, and mark the original residues: their scores should all be zero.

Does a positive score prove that a replacement improves protein function? Explain the difference between this LLR and rescoring the entire mutant sequence.


In [ ]:
# Read the scores for K and W at position 5.
# Plot heatmap[:, :10], label its columns, and mark the original residues.


## References

- [ESM-2 model card](https://huggingface.co/facebook/esm2_t6_8M_UR50D).
- [Checkpoint configuration](https://huggingface.co/facebook/esm2_t6_8M_UR50D/blob/main/config.json) and [token vocabulary](https://huggingface.co/facebook/esm2_t6_8M_UR50D/blob/main/vocab.txt).
- [ESM model classes and outputs](https://huggingface.co/docs/transformers/v4.57.2/en/model_doc/esm). The notebook uses Transformers 4.57.1; this adjacent-version reference documents the same output interface.
- [Protein localization dataset](https://huggingface.co/datasets/neuralbioinfo/Course-ProteinLoc).
- [Amino-acid codes](https://www.ddbj.nig.ac.jp/ddbj/code-e.html).
- [Meier et al. (2021), Language models enable zero-shot prediction of the effects of mutations on protein function](https://proceedings.neurips.cc/paper_files/paper/2021/file/f51338d736f95dd42427296047067694-Paper.pdf).
- [Supplementary methods, including pseudo-log-likelihood](https://proceedings.neurips.cc/paper_files/paper/2021/file/f51338d736f95dd42427296047067694-Supplemental.pdf).
